### reference: https://github.com/rasbt/LLMs-from-scratch/blob/c41a022/ch03/02_bonus_efficient-multihead-attention/mha-implementations.ipynb

In [69]:
import math
import torch
import torch.nn as nn
from torch.nn.attention.flex_attention import flex_attention, create_block_mask
print("torch version: ", torch.__version__)

torch.manual_seed(123)

# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")

# for now just checking on CPU
device = torch.device("cpu")
print(f"using device: {device}")

torch version:  2.11.0
using device: cpu


In [70]:
batch_size = 8
context_len = 1024
embed_dim = 512
embeddings = torch.randn((batch_size, context_len, embed_dim), device=device)
print(embeddings.shape)

torch.Size([8, 1024, 512])


### naive method of MHA

In [71]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_len, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out 
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1))
        # read more here: 
        # https://docs.pytorch.org/docs/2.14/generated/torch.triu.html
        # https://discuss.pytorch.org/t/what-does-register-buffer-do/121091
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x) 
        values = self.W_value(x)
        queries = self.W_query(x)
        attention_scores = queries@keys.transpose(1, 2)
        attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        denom = 1/(self.d_out)**0.5
        normalized_attention_scores = torch.softmax(attention_scores*denom, dim=-1)
        context_vectors = normalized_attention_scores@values 
        return context_vectors

class MHAWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_len, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_len, dropout, qkv_bias) for _ in range(num_heads)]
        ) # running a sequential for loop!
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)
    
    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)

mha_wrapper = MHAWrapper(
    d_in=embed_dim, 
    d_out=embed_dim//8, 
    context_len=context_len, 
    dropout=0.0, 
    num_heads=8, 
    qkv_bias=False
).to(device)

mha_wrapper_output = mha_wrapper(embeddings)
print(mha_wrapper_output.shape)

torch.Size([8, 1024, 512])


### parallel MHA

In [72]:
class MHAParallel(nn.Module):
    def __init__(self, d_in, d_out, context_len, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out%num_heads==0, "d_out must be divisible by num_heads"
        self.d_out = d_out 
        self.num_heads = num_heads 
        self.head_dim = d_out//num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        values = self.W_value(x)
        queries = self.W_query(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = queries@keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(mask_bool, -torch.inf)
        denom = 1/(self.head_dim)**0.5
        attention_weights = torch.softmax(attention_scores*denom, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vector = (attention_weights@values).transpose(1, 2)
        context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector

mha_parallel = MHAParallel(
    d_in=embed_dim, 
    d_out=embed_dim, 
    context_len=context_len, 
    dropout=0, 
    num_heads=8, 
    qkv_bias=False
).to(device)

mha_parallel_output = mha_parallel(embeddings)
print(mha_parallel_output.shape)

torch.Size([8, 1024, 512])


### combinedQKV MHA

In [73]:
class MHACombinedQKV(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_len, dropout=0, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.context_len = context_len
        self.head_dim = d_out//num_heads
        self.qkv = nn.Linear(d_in, 3*d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape
        qkv = self.qkv(x)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv.unbind(0)
        attention_scores = queries@keys.transpose(-2, -1)
        attention_scores = attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        denom = 1/(self.head_dim)**0.5
        attention_weights = torch.softmax(attention_scores*denom, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vector = attention_weights@values 
        context_vector = context_vector.transpose(1, 2)
        context_vector = context_vector.contiguous().view(batch_size, num_tokens, embed_dim)
        context_vector = self.proj(context_vector)
        return context_vector 

mha_combined_qkv = MHACombinedQKV(
    d_in=embed_dim, 
    d_out=embed_dim, 
    context_len=context_len, 
    dropout=0,
    num_heads=8,
    qkv_bias=False
).to(device)
mha_combined_qkv_output = mha_combined_qkv(embeddings)
print(mha_combined_qkv_output.shape)

# torch.unbind(0) : 
# a = torch.tensor([[1, 2, 3],
#                   [4, 5, 6],
#                   [7, 8, 9]])
# print(a)
# tensor([[1, 2, 3],
#         [4, 5, 6],
#         [7, 8, 9]])
# x, y, z = torch.unbind(a, dim=0)
# print(x)
# print(y)
# print(z)
# tensor([1, 2, 3])
# tensor([4, 5, 6])
# tensor([7, 8, 9])

torch.Size([8, 1024, 512])


### scaledDot MHA

In [74]:
class MHAscaled(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_len, dropout=0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "issue here d_out not divisible by num_heads"
        self.num_heads = num_heads 
        self.context_len = context_len
        self.head_dim = d_out//num_heads 
        self.d_out = d_out 
        self.qkv = nn.Linear(d_in, 3*d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape 
        qkv = self.qkv(x)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv.unbind(0)
        if not self.training:
            use_dropout = 0 
        else:
            use_dropout = self.dropout 
        context_vector = nn.functional.scaled_dot_product_attention(queries, keys, values, attn_mask=None, dropout_p=use_dropout, is_causal=True)
        context_vector = context_vector.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)
        context_vector = self.proj(context_vector)
        return context_vector

mha_pytorch_scaled = MHAscaled(d_in=embed_dim,
                               d_out=embed_dim,
                               context_len=context_len,
                               dropout=0,
                               num_heads=8,
                               qkv_bias=False).to(device)
mha_pytorch_scaled_output = mha_pytorch_scaled(embeddings)
print(mha_pytorch_scaled_output.shape)

torch.Size([8, 1024, 512])


### torch MHA

In [75]:
class MHATorch(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_len, dropout=0, qkv_bias=False, need_weights=True):
        super().__init__()
        self.context_len = context_len 
        self.multihead_attention = nn.MultiheadAttention(
            embed_dim=d_out, 
            num_heads=num_heads,
            dropout=dropout,
            bias=qkv_bias,
            add_bias_kv=qkv_bias,
            batch_first=True
        )
        self.need_weights = need_weights 
        self.proj = nn.Linear(d_out, d_out)
        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, _ = x.shape
        if self.context_len>=num_tokens:
            attention_mask = self.mask[:num_tokens, :num_tokens]
        else:
            attention_mask = self.mask[:self.context_len, :self.context_len]
        attention_output, _ = self.multihead_attention(
            x, x, x, attn_mask=attention_mask, need_weights=self.need_weights
        )
        output = self.proj(attention_output)
        return output 

mha_pytorch_class_default = MHATorch(
    d_in=embed_dim,
    d_out=embed_dim,
    context_len=context_len,
    dropout=0.0,
    num_heads=8,
    qkv_bias=False
).to(device)

mha_pytorch_class_default_output = mha_pytorch_class_default(embeddings)
print(mha_pytorch_class_default_output.shape)

torch.Size([8, 1024, 512])


### nn.MultiheadAttention with scaled_dot_product_attention

In [76]:
mha_pytorch_class_default = MHATorch(
    d_in=embed_dim,
    d_out=embed_dim,
    context_len=context_len,
    dropout=0.0,
    num_heads=8,
    qkv_bias=False,
    need_weights=False
).to(device)

mha_pytorch_class_default_scaled_output = mha_pytorch_class_default(embeddings)
print(mha_pytorch_class_default_scaled_output.shape)

torch.Size([8, 1024, 512])


### torchFlexAttention

In [77]:
def causal(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

class MHAFlexAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_len, dropout=0, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.context_len = context_len 
        self.head_dim = d_out//num_heads 
        self.d_out = d_out 
        self.qkv = nn.Linear(d_in, 3*d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout 

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.shape 
        qkv = self.qkv(x)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv 
        attn_mask = create_block_mask(causal, B=None, H=None, Q_LEN=num_tokens, KV_LEN=num_tokens, device=x.device)
        context_vec = flex_attention(queries, keys, values, block_mask=attn_mask)
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)
        context_vec = self.proj(context_vec)
        return context_vec

# FlexAttention does not run on MPS, so use the CPU there. Local names only: the global `device` stays untouched.
# torch.device(device) accepts both a str and a torch.device, so a stale `device = "cpu"` can't break this line.
flex_device = torch.device("cpu") if torch.device(device).type == "mps" else torch.device(device)
flex_embeddings = embeddings.to(flex_device)

mha_flex = MHAFlexAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    context_len=context_len,
    dropout=0,
    num_heads=8,
    qkv_bias=False
).to(flex_device)

# CPU FlexAttention has no backward pass, so run forward-only without gradients
with torch.no_grad():
    mha_flex_output = mha_flex(flex_embeddings)
print(mha_flex_output.shape)

torch.Size([8, 1024, 512])


In [78]:
print(mha_wrapper_output[0, 0, :10])
print(mha_parallel_output[0, 0, :10])
print(mha_combined_qkv_output[0, 0, :10])
print(mha_pytorch_scaled_output[0, 0, :10])
print(mha_pytorch_class_default_output[0, 0, :10])
print(mha_pytorch_class_default_scaled_output[0, 0, :10])
print(mha_flex_output[0, 0, :10])

# all are different because all of them have different initilization

tensor([ 0.0450, -0.0126, -0.0273, -0.6594,  0.1064, -0.1269,  0.0808, -0.0451,
        -0.0712,  0.0900], grad_fn=<SliceBackward0>)
tensor([ 0.0258,  0.2095, -0.5484, -0.0884,  0.3965,  0.0744,  0.2138,  0.0675,
        -0.5376, -0.3985], grad_fn=<SliceBackward0>)
tensor([-0.2442, -0.7437,  0.2165, -0.1120,  0.1441, -0.1398, -0.0075, -0.2092,
        -0.6045, -0.1457], grad_fn=<SliceBackward0>)
tensor([-0.1636,  0.4923, -0.4310,  0.3487, -0.0809,  0.0304,  0.3023,  0.6196,
         0.0854, -0.2341], grad_fn=<SliceBackward0>)
tensor([ 0.3983,  0.5185, -0.3980, -0.1116,  0.2220, -0.0748, -0.0762,  0.3711,
         0.2712,  0.3635], grad_fn=<SliceBackward0>)
tensor([ 0.2398, -0.6029,  0.0710, -0.0515, -0.1654, -0.1303, -0.4796, -0.1328,
         0.0263,  0.5456], grad_fn=<SliceBackward0>)
tensor([-0.1975, -0.2319, -0.2777, -0.7564,  0.7220,  0.4797, -0.5518, -0.1089,
         0.0423, -0.0810])


In [79]:
# tensor([ 0.2056,  0.1216,  0.3188, -0.1244, -0.2744, -0.3650,  0.1341,  0.1663,
#         -0.7210,  0.3889], device='mps:0', grad_fn=<SliceBackward0>)
# tensor([ 0.1137,  0.3047, -0.0449,  0.4943, -0.1345,  0.2653, -0.0926, -0.6561,
#          0.0682, -0.4110], device='mps:0', grad_fn=<SliceBackward0>)
# tensor([ 0.4443, -0.2871, -0.0815, -0.1258, -0.0581, -0.7157,  0.0366,  0.1140,
#          0.0227, -0.2649], device='mps:0', grad_fn=<SliceBackward0>)
# tensor([ 0.4870, -0.3473, -0.6062, -0.3920,  0.3861,  0.1917,  0.0320, -0.1301,
#         -0.5004, -0.1106], device='mps:0', grad_fn=<SliceBackward0>)
# tensor([-0.0661, -0.1910, -0.3417, -0.1036, -0.2394, -0.1800, -0.0382, -0.0679,
#          0.1795,  0.3542], device='mps:0', grad_fn=<SliceBackward0>)
# tensor([-0.3400,  0.3846,  0.0014, -0.4655,  0.0300,  0.0461,  0.0571, -0.0010,
#          0.1947,  0.0229], device='mps:0', grad_fn=<SliceBackward0>)
# tensor([-0.1916, -0.1909, -0.1470, -0.1559, -0.4098, -0.1114,  0.2146,  0.0359,
#          0.5498,  0.0845])

In [80]:
%timeit -n 10 -r 7 mha_wrapper(embeddings)

132 ms ± 7.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%timeit -n 10 -r 7 mha_parallel(embeddings)

In [ ]:
%timeit -n 10 -r 7 mha_combined_qkv(embeddings)

124 ms ± 4.09 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%timeit -n 10 -r 7 mha_pytorch_scaled(embeddings)

33.5 ms ± 1.52 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%timeit -n 10 -r 7 mha_pytorch_class_default(embeddings)

64.8 ms ± 4.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%timeit -n 10 -r 7
with torch.no_grad():
    mha_flex(flex_embeddings)

218 ms ± 9.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
